# Kaggriculture Agent | Fixed Trace + Safe Selling

This notebook releases a deterministic, readable Kaggriculture learning agent built around one
simple idea: replay a fixed 719-action plan, but repair two common failure modes from the live
observation before returning an action.

It is deliberately an older and materially weaker public baseline, not the current private
competition agent. The exact public source lost all 32 local games against the two stronger
private-source references used for separation, across eight consumed seeds and both seats under
`kaggle-environments==1.32.6`. That is local head-to-head evidence, not a ladder rating estimate.

**Artifact identity:** `81159A18DF3CD1A5CAABE5EDBBA0A597561775F77A49465FA39DF6802A706922` (SHA-256). The title contains no score claim,
and this release makes no promise about a Kaggle rating.


## 1. The useful mental model: plan plus guardrail

A fixed trace is an **open-loop plan**: at turn `t`, return the action recorded for turn `t`.
That makes it deterministic and easy to audit, but brittle when the opponent or engine changes
the state. This baseline adds a narrow **closed-loop safety layer**:

| Layer | Reads | Does | Does not do |
|---|---|---|---|
| Schedule | current step | chooses the recorded farmer, hand, and market actions | search, plan, or model the opponent |
| Hand normalizer | current number of hands | truncates extra actions and pads missing ones with `PASS` | reassign work intelligently |
| Sell guard | shed plus same-turn shed movement | clamps or removes impossible `SELL` orders | forecast price or optimize order size |

This pattern is useful beyond Kaggriculture: keep a simple policy legible, then put state-aware
checks at the interface where invalid assumptions become costly.


## 2. Reading the agent from the bottom up

The source is large because `SCHEDULE` stores 719 explicit actions. The reusable logic is small:

1. `_expected_hands` reads the live farm and determines how many hand actions are legal.
2. `_normalize_hands` keeps the scheduled list aligned with that live count.
3. `_predicted_shed` starts from observed storage, then accounts for same-turn `DROP` and
   `PICKUP` actions made on shed tiles.
4. `_guard_sells` processes sell orders in order, clamping each to the remaining predicted stock.
5. `agent` selects the scheduled step, applies the guards, and returns the three action channels.

Notice the ordering: storage is predicted **after** normalizing unit actions and **before**
validating market sales. A guard built from the pre-action shed alone would reject valid same-turn
deposits or allow sales after a pickup removed the stock.


## 3. Provenance and license

This is not presented as a newly invented trajectory.

- Upstream notebook: [Kaggriculture Frontier Lab | High-Score + Visuals](https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-lab-high-score-visuals) by PRVSIYAN.
- Upstream license displayed on Kaggle: Apache License 2.0.
- Teacher evidence: public episode `89256171`, team `radiant-allomancer`, player `0`.
- Replay SHA-256: `ADADDE60EEFB93B56DBE9922B51D5D1970803F89DDF0EEEE67B9890024090405`.
- Local modifications: independent schedule regeneration, observation-based hand normalization,
  and impossible-sell drift guards.

The exact attribution stays in `main.py`. The release bundle also carries the Apache-2.0 license
and a NOTICE file. Please retain the attribution when forking.


## 4. Why the engine version belongs in every result

An exact same-cell local diagnostic compared versions 1.32.2 and 1.32.6 on 20 games. Every final
reward tuple changed, and 16 of 20 win/tie/loss outcomes changed. The only resolved default
configuration change was `townCenterSellInterval: 12 -> 24`.

That diagnostic used already-consumed development seeds, so it is evidence of **engine
sensitivity**, not promotion evidence. It teaches three practical rules:

1. Pin `kaggle-environments==1.32.6` when publishing a current benchmark.
2. Record the resolved environment configuration, not just the package version.
3. Never carry a score or win rate across engine versions as if it measured the same policy game.

A historical remote submission record cannot repair missing byte provenance. Kaggle does not
expose the old uploaded archive here, so this release does not attach any historical ladder score
to the exact source below.


In [ ]:
%%writefile main.py
# Derived from "Kaggriculture Frontier Lab | High-Score + Visuals" by PRVSIYAN.
# Source: https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-lab-high-score-visuals
# Upstream license: Apache-2.0, as displayed on the public Kaggle Competition Notebook.
# Modifications: independently regenerated from the attributed public replay and added
# observation-based hand normalization plus impossible-SELL drift guards.
# Fixed schedule extracted from a Kaggriculture replay.
# Source episode: 89256171
# Source team: radiant-allomancer
# Source player: 0
# Source replay SHA256: ADADDE60EEFB93B56DBE9922B51D5D1970803F89DDF0EEEE67B9890024090405
from copy import deepcopy

SOURCE_EPISODE_ID = 89256171
SOURCE_TEAM = 'radiant-allomancer'
SOURCE_PLAYER = 0
SOURCE_REPLAY_SHA256 = 'ADADDE60EEFB93B56DBE9922B51D5D1970803F89DDF0EEEE67B9890024090405'
RECORDED_ACTIONS = 719
EPISODE_STEPS = 720
SCHEDULE = [{'farmer': ['PASS'],
  'hands': [],
  'market': [['HIRE'], ['HIRE'], ['BUY_ANIMAL', 'COW', 3], ['BUY_SEED', 'MELON', 6]]},
 {'farmer': ['BUILD_PASTURE'],
  'hands': [['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['BUY_ANIMAL', 'SHEEP', 1]]},
 {'farmer': ['PICKUP', 'COW', 1],
  'hands': [['PICKUP', 'SHEEP', 1], ['NORTH'], ['WEST']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2], ['HIRE']]},
 {'farmer': ['PLACE', 'COW'],
  'hands': [['WEST'], ['BUILD_PASTURE'], ['NORTH'], ['WEST']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['BUILD_PASTURE'], ['NORTH'], ['SOUTH'], ['PICKUP', 'COW', 1]],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'], ['BUILD_PASTURE'], ['PICKUP', 'COW', 1], ['CARE']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['PASS'],
  'hands': [['NORTH'], ['NORTH'], ['NORTH'], ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['EAST'], ['NORTH'], ['PLACE', 'COW'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['PLACE', 'SHEEP'], ['PLANT', 'MELON'], ['CARE'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['WATER'], ['NORTH'], ['SOUTH']],
  'market': [['SELL', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['SOUTH'], ['WEST'], ['SOUTH'], ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'], ['PLANT', 'MELON'], ['NORTH'], ['PLACE', 'COW']],
  'market': [['SELL', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['NORTH'], ['WATER'], ['NORTH'], ['CARE']],
  'market': [['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['SOUTH'],
  'hands': [['PLANT', 'MELON'], ['WEST'], ['SOUTH'], ['EAST']],
  'market': [['SELL', 'WHEAT', 3]]},
 {'farmer': ['WEST'],
  'hands': [['WATER'], ['PLANT', 'MELON'], ['NORTH'], ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'], ['WATER'], ['SOUTH'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3]]},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'], ['WEST'], ['SOUTH'], ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 4]]},
 {'farmer': ['PASS'],
  'hands': [['NORTH'], ['PLANT', 'MELON'], ['SOUTH'], ['PASS']],
  'market': [['SELL', 'WHEAT', 4]]},
 {'farmer': ['PASS'],
  'hands': [['WEST'], ['WATER'], ['PASS'], ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 4]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'], ['WEST'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 4]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'], ['PLANT', 'MELON'], ['PASS'], ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 4]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'], ['WATER'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 4]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 4]]},
 {'farmer': ['PASS'],
  'hands': [['EAST'], ['SOUTH'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 4]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['BUY_PRODUCT', 'WHEAT', 4], ['HIRE'], ['HIRE']]},
 {'farmer': ['FEED'], 'hands': [['WEST'], ['NORTH']], 'market': [['HIRE'], ['HIRE']]},
 {'farmer': ['CARE'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['HIRE']]},
 {'farmer': ['NORTH'],
  'hands': [['WEST'], ['CARE'], ['NORTH'], ['NORTH'], ['WEST']],
  'market': [['SELL', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['COLLECT_FERTILIZER'], ['NORTH'], ['NORTH'], ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['COLLECT_FERTILIZER'], ['SOUTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['EAST']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'], ['DROP'], ['SOUTH'], ['SOUTH'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['DROP'], ['PASS'], ['SOUTH'], ['SOUTH'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 2], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['DROP'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2], ['BUY_SEED', 'MELON', 1]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['NORTH'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PLANT', 'MELON'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['SOUTH'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['SOUTH'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 3], ['HIRE'], ['HIRE']]},
 {'farmer': ['CARE'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['HIRE']]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'], ['CARE'], ['WEST'], ['NORTH'], ['WEST']],
  'market': [['SELL', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['NORTH'], ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['EAST']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'], ['SOUTH'], ['EAST'], ['SOUTH'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['WATER'], ['SOUTH'], ['DROP'], ['SOUTH'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['SOUTH'], ['PASS'], ['PASS'], ['DROP'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['NORTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2], ['BUY_SEED', 'MELON', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['NORTH'], ['NORTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['NORTH'], ['NORTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['NORTH'], ['NORTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['PLANT', 'MELON'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['EAST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'FERTILIZER', 2],
             ['SELL', 'WHEAT', 3],
             ['BUY_PRODUCT', 'WHEAT', 1],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 3], ['HIRE'], ['HIRE'], ['BUY_SEED', 'MELON', 2]]},
 {'farmer': ['CARE'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['HIRE']]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'], ['CARE'], ['WEST'], ['NORTH'], ['WEST']],
  'market': [['SELL', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['NORTH'], ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'], ['CARE'], ['COLLECT_FERTILIZER'], ['COLLECT_FERTILIZER'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'], ['NORTH'], ['EAST'], ['SOUTH'], ['SOUTH']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'], ['WEST'], ['DROP'], ['SOUTH'], ['EAST']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'], ['WEST'], ['PASS'], ['DROP'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['WEST'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WEST'], ['WATER'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['WEST'], ['WEST'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PLANT', 'MELON'], ['PLANT', 'MELON'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['WATER'], ['WATER'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['EAST'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['EAST'], ['EAST'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['EAST'], ['EAST'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['EAST'], ['EAST'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['DROP'], ['EAST'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 2], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 3], ['HIRE'], ['HIRE']]},
 {'farmer': ['CARE'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'], ['CARE'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['SOUTH']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'], ['WEST'], ['EAST'], ['SOUTH'], ['SOUTH'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['WATER'], ['WEST'], ['DROP'], ['SOUTH'], ['EAST'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['SOUTH'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['DROP'], ['PASS'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['NORTH'], ['WATER'], ['NORTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['WEST'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['NORTH'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['NORTH'], ['NORTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['NORTH'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['DROP'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 2], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['NORTH'], ['NORTH'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['EAST'],
  'hands': [['NORTH'], ['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WEST'], ['WEST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 3], ['HIRE'], ['HIRE'], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['CARE'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'], ['CARE'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'], ['WEST'], ['EAST'], ['WEST'], ['WEST'], ['SOUTH']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['PLANT', 'STRAWBERRY'],
            ['DROP'],
            ['WEST'],
            ['PLANT', 'STRAWBERRY'],
            ['WEST']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'], ['WATER'], ['PASS'], ['WEST'], ['WATER'], ['WEST']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['WEST'], ['WEST'], ['SOUTH']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'], ['SOUTH'], ['PASS'], ['WATER'], ['PLANT', 'STRAWBERRY'], ['EAST']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'], ['EAST'], ['PASS'], ['NORTH'], ['WATER'], ['EAST']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'], ['PASS'], ['PASS'], ['WATER'], ['NORTH'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'], ['PASS'], ['PASS'], ['EAST'], ['SOUTH'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['EAST'], ['PASS'], ['PASS'], ['WATER'], ['SOUTH'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['DROP'], ['PASS'], ['PASS'], ['SOUTH'], ['EAST'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 2], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['EAST'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['WEST'], ['EAST'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PLANT', 'STRAWBERRY'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['WATER'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['SOUTH'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WHEAT', 3],
             ['SELL', 'FERTILIZER', 1],
             ['BUY_PRODUCT', 'WHEAT', 1],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 3], ['HIRE'], ['HIRE'], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['CARE'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'], ['CARE'], ['WEST'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['CARE'], ['NORTH'], ['NORTH'], ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'], ['HARVEST'], ['COLLECT_FERTILIZER'], ['CARE'], ['NORTH'], ['WEST']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['EAST'], ['SOUTH'], ['SOUTH'], ['WEST']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['SOUTH'], ['WATER'], ['DROP'], ['SOUTH'], ['SOUTH'], ['WEST']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['SOUTH'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['PLANT', 'STRAWBERRY']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['DROP'], ['WEST'], ['PASS'], ['PASS'], ['PASS'], ['WATER']],
  'market': [['SELL', 'FERTILIZER', 3], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['PASS'], ['WEST'], ['PASS'], ['PASS'], ['PASS'], ['EAST']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['WATER'],
  'hands': [['NORTH'], ['WATER'], ['NORTH'], ['PASS'], ['PASS'], ['PLANT', 'STRAWBERRY']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['WEST'], ['SOUTH'], ['PASS'], ['PASS'], ['WATER']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['NORTH'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'], ['NORTH'], ['PASS'], ['PASS'], ['PASS'], ['SOUTH']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['PASS'], ['WATER'], ['PASS'], ['PASS'], ['PASS'], ['SOUTH']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WATER'],
  'hands': [['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['EAST']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['EAST']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['PASS'], ['EAST']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['EAST'],
  'hands': [['PASS'], ['DROP'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WOOL', 6], ['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2], ['BUY_ANIMAL', 'COW', 2]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 5], ['HIRE'], ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]},
 {'farmer': ['CARE'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2], ['HIRE'], ['HIRE']]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'], ['CARE'], ['NORTH'], ['NORTH'], ['WEST'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'], ['NORTH'], ['WEST'], ['WEST'], ['WEST'], ['WEST']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['WEST'], ['CARE'], ['BUILD_PASTURE'], ['NORTH'], ['CARE'], ['COLLECT_FERTILIZER']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['COLLECT_FERTILIZER'], ['NORTH'], ['WATER'], ['WEST'], ['EAST']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 2], ['HIRE']]},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['BUILD_PASTURE'],
            ['PICKUP', 'COW', 1],
            ['WEST']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'], ['WATER'], ['WATER'], ['WEST'], ['EAST'], ['PICKUP', 'COW', 1], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['WATER'], ['NORTH'], ['WEST'], ['WEST'], ['EAST'], ['NORTH'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['WATER'], ['WEST'], ['WATER'], ['PASS'], ['WEST'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['EAST'],
  'hands': [['WATER'], ['WEST'], ['WATER'], ['NORTH'], ['PASS'], ['PLACE', 'COW'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 4]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['SOUTH'], ['WATER'], ['NORTH'], ['WATER'], ['PASS'], ['CARE'], ['PASS']],
  'market': [['SELL', 'WHEAT', 4], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['EAST'], ['WEST'], ['WATER'], ['NORTH'], ['PASS'], ['SOUTH'], ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 4]]},
 {'farmer': ['WEST'],
  'hands': [['EAST'], ['WATER'], ['SOUTH'], ['WATER'], ['PASS'], ['WEST'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['PASS'], ['PLACE', 'COW'], ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['EAST'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['PASS'], ['CARE'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['DROP'], ['SOUTH'], ['SOUTH'], ['SOUTH'], ['PASS'], ['EAST'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 2], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['PASS'], ['SOUTH'], ['EAST'], ['SOUTH'], ['PASS'], ['EAST'], ['PASS']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['PASS'], ['EAST'], ['EAST'], ['EAST'], ['PASS'], ['DROP'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['EAST'],
  'hands': [['PASS'], ['EAST'], ['EAST'], ['EAST'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['DROP'], ['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1], ['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS'], ['PASS']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 3]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WHEAT', 5], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 5], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['HARVEST'], ['CARE'], ['WEST'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'], ['WEST'], ['NORTH'], ['WEST'], ['WEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['WEST'], ['HARVEST'], ['HARVEST'], ['CARE'], ['NORTH'], ['NORTH'], ['WEST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 1],
  'hands': [['COLLECT_FERTILIZER'],
            ['WEST'],
            ['CARE'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['WEST'], ['CARE'], ['NORTH'], ['CARE'], ['NORTH'], ['CARE'], ['NORTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['WEST'], ['WATER'], ['SOUTH'], ['SOUTH'], ['WEST'], ['WEST'], ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'], ['EAST'], ['SOUTH'], ['EAST'], ['WATER'], ['WEST'], ['WEST']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'], ['EAST'], ['DROP'], ['DROP'], ['NORTH'], ['WATER'], ['WATER']],
  'market': [['SELL', 'MILK', 6], ['SELL', 'FERTILIZER', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['EAST'], ['EAST'], ['PASS'], ['PASS'], ['WATER'], ['SOUTH'], ['NORTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['EAST'], ['NORTH'], ['PASS'], ['PASS'], ['NORTH'], ['SOUTH'], ['WATER']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['EAST'], ['SOUTH'], ['PASS'], ['PASS'], ['WATER'], ['NORTH'], ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'], ['DROP'], ['PASS'], ['PASS'], ['WEST'], ['SOUTH'], ['WATER']],
  'market': [['SELL', 'MILK', 5]]},
 {'farmer': ['WEST'],
  'hands': [['DROP'], ['PASS'], ['PASS'], ['PASS'], ['WATER'], ['EAST'], ['WEST']],
  'market': [['SELL', 'MILK', 6],
             ['SELL', 'FERTILIZER', 3],
             ['BUY_PRODUCT', 'WHEAT', 1],
             ['BUY_LAND']]},
 {'farmer': ['FEED'],
  'hands': [['PASS'], ['PASS'], ['PASS'], ['PASS'], ['WEST'], ['EAST'], ['SOUTH']],
  'market': [['BUY_ANIMAL', 'SHEEP', 3], ['BUY_SEED', 'STRAWBERRY', 6]]},
 {'farmer': ['EAST'],
  'hands': [['EAST'], ['NORTH'], ['EAST'], ['PASS'], ['WATER'], ['EAST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 7]]},
 {'farmer': ['EAST'],
  'hands': [['BUILD_PASTURE'], ['EAST'], ['EAST'], ['PASS'], ['SOUTH'], ['PASS'], ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'SHEEP', 1],
  'hands': [['PASS'],
            ['BUILD_PASTURE'],
            ['BUILD_PASTURE'],
            ['PASS'],
            ['SOUTH'],
            ['PASS'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'SHEEP', 2],
  'hands': [['PASS'], ['NORTH'], ['WEST'], ['PASS'], ['SOUTH'], ['PASS'], ['PLANT', 'STRAWBERRY']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['PASS'], ['NORTH'], ['PASS'], ['PASS'], ['SOUTH'], ['PASS'], ['WATER']],
  'market': []},
 {'farmer': ['PLACE', 'SHEEP'],
  'hands': [['PASS'], ['SOUTH'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['EAST']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['NORTH'], ['PASS'], ['PASS'], ['EAST'], ['PASS'], ['PLANT', 'STRAWBERRY']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WHEAT', 8], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['NORTH'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 8], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['PICKUP', 'SHEEP', 2],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 5], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['CARE'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 5]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['WEST'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['CARE'],
            ['NORTH'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['HARVEST'],
            ['CARE'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 3],
  'hands': [['WEST'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['CARE'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['CARE'],
            ['NORTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['EAST'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['EAST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['WEST'],
            ['PLACE', 'SHEEP'],
            ['WEST'],
            ['EAST'],
            ['WATER'],
            ['WATER'],
            ['NORTH'],
            ['EAST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['CARE'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 1],
            ['WATER']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['EAST'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['NORTH'],
            ['EAST'],
            ['WEST'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['NORTH'],
            ['FEED'],
            ['EAST']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['PLACE', 'SHEEP'],
            ['WATER'],
            ['NORTH'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['EAST'],
            ['DIG']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['NORTH'],
            ['CARE'],
            ['NORTH'],
            ['SOUTH'],
            ['PASS'],
            ['NORTH'],
            ['SOUTH'],
            ['PASS'],
            ['PLANT', 'STRAWBERRY']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WATER'],
            ['WEST'],
            ['WATER'],
            ['DROP'],
            ['PASS'],
            ['EAST'],
            ['NORTH'],
            ['PASS'],
            ['WATER']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['NORTH'],
            ['PASS'],
            ['SOUTH'],
            ['PASS'],
            ['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS'],
            ['EAST']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['PASS'],
            ['SOUTH'],
            ['PASS'],
            ['PASS'],
            ['NORTH'],
            ['SOUTH'],
            ['PASS'],
            ['PLANT', 'STRAWBERRY']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['PASS'],
            ['SOUTH'],
            ['PASS'],
            ['PASS'],
            ['EAST'],
            ['SOUTH'],
            ['PASS'],
            ['WATER']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'],
            ['PASS'],
            ['SOUTH'],
            ['PASS'],
            ['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['PASS'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['NORTH'],
            ['EAST'],
            ['PASS'],
            ['PLANT', 'STRAWBERRY']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['PASS'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['EAST'],
            ['EAST'],
            ['PASS'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['EAST'],
            ['PASS'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['SOUTH'],
            ['DROP'],
            ['PASS'],
            ['EAST']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['EAST'],
            ['PASS'],
            ['DROP'],
            ['PASS'],
            ['PASS'],
            ['SOUTH'],
            ['PASS'],
            ['PASS'],
            ['PLANT', 'STRAWBERRY']],
  'market': [['SELL', 'WOOL', 4]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WHEAT', 9], ['HIRE'], ['HIRE'], ['HIRE'], ['BUY_SEED', 'STRAWBERRY', 3]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['NORTH'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 9], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'], ['CARE'], ['NORTH'], ['NORTH'], ['NORTH'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 5], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['HARVEST'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 5], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['CARE'],
            ['CARE'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['HARVEST'],
            ['NORTH'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 3],
  'hands': [['WEST'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['CARE'],
            ['CARE'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['CARE'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WEST']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['WEST'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['EAST'],
            ['WEST'],
            ['WATER'],
            ['EAST'],
            ['EAST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['WEST'],
            ['WEST'],
            ['HARVEST'],
            ['PLANT', 'STRAWBERRY'],
            ['EAST'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 1],
            ['DROP'],
            ['NORTH']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['WATER'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['NORTH'],
            ['WATER'],
            ['WEST'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['EAST'],
            ['HARVEST'],
            ['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH'],
            ['FEED'],
            ['SOUTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['EAST'],
            ['PLANT', 'STRAWBERRY'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['WATER'],
            ['EAST'],
            ['PICKUP', 'WHEAT', 1],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['NORTH'],
            ['EAST'],
            ['WATER'],
            ['WATER'],
            ['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['HARVEST'],
            ['DROP'],
            ['EAST'],
            ['EAST']],
  'market': [['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WATER'],
            ['EAST'],
            ['EAST'],
            ['HARVEST'],
            ['WATER'],
            ['EAST'],
            ['WEST'],
            ['PASS'],
            ['FEED'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['NORTH'],
            ['DROP'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['DROP'],
            ['WATER'],
            ['PASS'],
            ['PASS'],
            ['SOUTH']],
  'market': [['SELL', 'MILK', 6], ['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['DIG'],
            ['PASS'],
            ['HARVEST'],
            ['PASS'],
            ['PASS'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_ANIMAL', 'COW', 1], ['BUY_SEED', 'MELON', 6]]},
 {'farmer': ['EAST'],
  'hands': [['HARVEST'],
            ['NORTH'],
            ['BUILD_PASTURE'],
            ['SOUTH'],
            ['PLANT', 'MELON'],
            ['PASS'],
            ['SOUTH'],
            ['PASS'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['PLANT', 'MELON'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['PICKUP', 'COW', 1],
            ['SOUTH'],
            ['PASS'],
            ['SOUTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['PASS'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['PASS'],
            ['PASS'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['EAST'],
            ['PASS'],
            ['SOUTH'],
            ['DROP'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['PASS'],
            ['PASS'],
            ['WEST']],
  'market': [['SELL', 'MELON', 12], ['SELL', 'FERTILIZER', 2]]},
 {'farmer': ['WEST'],
  'hands': [['PLANT', 'MELON'],
            ['NORTH'],
            ['SOUTH'],
            ['PASS'],
            ['PLANT', 'MELON'],
            ['EAST'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['PLANT', 'MELON']],
  'market': [['BUY_ANIMAL', 'COW', 3], ['BUY_SEED', 'MELON', 2]]},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['WATER'],
            ['PLACE', 'COW'],
            ['EAST'],
            ['NORTH'],
            ['EAST'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 6], ['BUY_SEED', 'STRAWBERRY', 3]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'MELON', 24],
             ['SELL', 'MILK', 3],
             ['SELL', 'WHEAT', 14],
             ['SELL', 'FERTILIZER', 3],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['NORTH'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 12], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['PICKUP', 'COW', 1],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 8], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['CARE'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 8], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['EAST'],
            ['WEST'],
            ['CARE'],
            ['CARE'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['BUILD_PASTURE'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['EAST'],
            ['WEST'],
            ['WEST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['NORTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['BUILD_PASTURE'],
            ['CARE'],
            ['EAST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['COLLECT_FERTILIZER'],
            ['PICKUP', 'COW', 2],
            ['SOUTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 1],
  'hands': [['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['WEST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['EAST'],
            ['NORTH'],
            ['WATER'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['DROP'],
            ['DIG'],
            ['EAST'],
            ['WEST'],
            ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['EAST'],
            ['WEST'],
            ['PLANT', 'STRAWBERRY'],
            ['WEST'],
            ['PASS'],
            ['NORTH'],
            ['DROP'],
            ['PLACE', 'COW'],
            ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'],
            ['HARVEST'],
            ['WATER'],
            ['WATER'],
            ['WATER'],
            ['PLANT', 'STRAWBERRY'],
            ['NORTH'],
            ['PLANT', 'MELON'],
            ['NORTH'],
            ['CARE'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['EAST'],
            ['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['EAST']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['PICKUP', 'WHEAT', 1],
            ['SOUTH'],
            ['PASS'],
            ['EAST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['PLANT', 'MELON'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH'],
            ['WATER'],
            ['WATER'],
            ['WEST'],
            ['PLANT', 'MELON'],
            ['PASS'],
            ['EAST'],
            ['WEST']],
  'market': [['BUY_LAND']]},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['WEST'],
            ['FEED'],
            ['WATER'],
            ['PASS'],
            ['EAST'],
            ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 6]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['PICKUP', 'WHEAT', 2],
            ['EAST'],
            ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['PLANT', 'STRAWBERRY'],
            ['PLACE', 'COW'],
            ['PLANT', 'STRAWBERRY'],
            ['WATER'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['PLANT', 'STRAWBERRY'],
            ['NORTH'],
            ['PLACE', 'COW'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['CARE'],
            ['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['EAST'],
            ['EAST'],
            ['WEST'],
            ['WEST'],
            ['CARE'],
            ['SOUTH']],
  'market': [['BUY_SEED', 'STRAWBERRY', 3]]},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['PLANT', 'STRAWBERRY'],
            ['PASS'],
            ['WEST'],
            ['FEED'],
            ['WEST'],
            ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['PLANT', 'STRAWBERRY'],
            ['NORTH'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['PASS'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['WEST'],
            ['PASS'],
            ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['PLANT', 'STRAWBERRY'],
            ['DROP'],
            ['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['PASS'],
            ['FEED'],
            ['PASS'],
            ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['DROP'],
            ['PLANT', 'STRAWBERRY'],
            ['WEST'],
            ['WATER'],
            ['NORTH'],
            ['WATER'],
            ['SOUTH'],
            ['PASS'],
            ['SOUTH'],
            ['PASS'],
            ['PASS']],
  'market': [['SELL', 'FERTILIZER', 3],
             ['BUY_PRODUCT', 'WHEAT', 2],
             ['BUY_SEED', 'STRAWBERRY', 2]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'MELON', 6],
             ['SELL', 'WHEAT', 12],
             ['SELL', 'FERTILIZER', 3],
             ['HIRE'],
             ['HIRE'],
             ['HIRE'],
             ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['NORTH'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 12], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['PICKUP', 'WHEAT', 4],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 8], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['CARE'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 8], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['CARE'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['FEED'],
            ['HARVEST'],
            ['CARE'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 2],
  'hands': [['NORTH'],
            ['FEED'],
            ['CARE'],
            ['HARVEST'],
            ['NORTH'],
            ['CARE'],
            ['CARE'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WEST']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['PLANT', 'STRAWBERRY'],
            ['EAST'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['CARE'],
            ['NORTH'],
            ['EAST'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['FEED'],
            ['WEST'],
            ['WEST'],
            ['WATER'],
            ['EAST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH']],
  'market': [['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['DIG'],
            ['FEED'],
            ['WEST'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['NORTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['WATER'],
            ['WEST'],
            ['NORTH'],
            ['CARE'],
            ['EAST'],
            ['WEST'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['WATER'],
            ['WEST']],
  'market': [['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['PICKUP', 'WHEAT', 4],
            ['SOUTH'],
            ['FERTILIZE'],
            ['WEST'],
            ['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['HARVEST'],
            ['WEST']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['EAST'],
            ['EAST'],
            ['PLANT', 'STRAWBERRY'],
            ['WATER'],
            ['WATER'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH'],
            ['PLANT', 'STRAWBERRY'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WATER'],
            ['FEED'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['DROP'],
            ['NORTH'],
            ['WATER'],
            ['PASS']],
  'market': [['SELL', 'FERTILIZER', 3], ['BUY_SEED', 'STRAWBERRY', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['EAST'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WATER'],
            ['FEED'],
            ['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['PASS'],
            ['NORTH'],
            ['SOUTH'],
            ['PASS']],
  'market': [['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['EAST'],
  'hands': [['NORTH'],
            ['EAST'],
            ['WATER'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['DROP'],
            ['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS']],
  'market': [['SELL', 'MILK', 3], ['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['FEED'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['PLANT', 'STRAWBERRY'],
            ['PASS'],
            ['PASS'],
            ['SOUTH'],
            ['EAST'],
            ['PASS']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'],
            ['WEST'],
            ['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['PASS'],
            ['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['EAST'],
            ['EAST'],
            ['SOUTH'],
            ['PASS'],
            ['PASS'],
            ['WEST'],
            ['EAST'],
            ['PASS']],
  'market': [['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'],
            ['PASS'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['PLANT', 'STRAWBERRY'],
            ['PASS'],
            ['PASS'],
            ['WEST'],
            ['DROP'],
            ['PASS']],
  'market': [['SELL', 'MELON', 6], ['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'MILK', 6],
             ['SELL', 'WOOL', 4],
             ['SELL', 'FERTILIZER', 7],
             ['SELL', 'WHEAT', 12],
             ['HIRE'],
             ['HIRE'],
             ['HIRE'],
             ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['NORTH'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 12], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 8], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['CARE'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 8], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['CARE'],
            ['WEST'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['FEED'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['CARE'],
            ['CARE'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['CARE'],
            ['CARE'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 2],
  'hands': [['EAST'],
            ['FEED'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['EAST'],
            ['EAST'],
            ['NORTH'],
            ['WEST'],
            ['WATER'],
            ['WEST'],
            ['SOUTH'],
            ['CARE'],
            ['CARE'],
            ['WATER'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['FEED'],
            ['NORTH'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['FEED'],
            ['EAST'],
            ['FERTILIZE'],
            ['WEST'],
            ['WEST'],
            ['PLANT', 'STRAWBERRY'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['WATER'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['HARVEST'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['FERTILIZE'],
            ['SOUTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['PICKUP', 'WHEAT', 4],
            ['WATER'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['PLANT', 'STRAWBERRY'],
            ['EAST'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['HARVEST'],
            ['WATER'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WEST'],
            ['FEED'],
            ['SOUTH'],
            ['SOUTH'],
            ['HARVEST'],
            ['PLANT', 'STRAWBERRY'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['PLANT', 'STRAWBERRY'],
            ['WATER'],
            ['PLANT', 'STRAWBERRY'],
            ['CARE'],
            ['EAST'],
            ['SOUTH'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1], ['BUY_SEED', 'MELON', 1]]},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'],
            ['FEED'],
            ['SOUTH'],
            ['DIG'],
            ['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['SOUTH'],
            ['WEST']],
  'market': [['BUY_SEED', 'STRAWBERRY', 2]]},
 {'farmer': ['EAST'],
  'hands': [['WATER'],
            ['EAST'],
            ['EAST'],
            ['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['FEED'],
            ['WATER'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['PLANT', 'STRAWBERRY'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS']],
  'market': [['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['WEST'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['WATER'],
            ['DROP'],
            ['SOUTH'],
            ['NORTH'],
            ['PASS']],
  'market': [['SELL', 'FERTILIZER', 3],
             ['BUY_PRODUCT', 'WHEAT', 2],
             ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['WEST'],
            ['WEST'],
            ['PLANT', 'STRAWBERRY'],
            ['EAST'],
            ['EAST'],
            ['EAST'],
            ['PASS'],
            ['NORTH'],
            ['EAST'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['DROP'],
            ['PASS'],
            ['WEST'],
            ['WATER'],
            ['EAST'],
            ['EAST'],
            ['PLANT', 'STRAWBERRY'],
            ['PASS'],
            ['SOUTH'],
            ['EAST'],
            ['PASS']],
  'market': [['SELL', 'FERTILIZER', 3], ['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'MELON', 12],
             ['SELL', 'WHEAT', 12],
             ['SELL', 'STRAWBERRY', 2],
             ['SELL', 'FERTILIZER', 4],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['HARVEST'], ['NORTH'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 12], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['PICKUP', 'WHEAT', 4],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 8], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['PICKUP', 'FERTILIZER', 3],
            ['NORTH'],
            ['WEST'],
            ['EAST'],
            ['CARE'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 8], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['CARE'],
            ['EAST'],
            ['CARE'],
            ['SOUTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['FEED'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['HARVEST'],
            ['HARVEST'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 1],
  'hands': [['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['CARE']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 1],
  'hands': [['WATER'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['CARE'],
            ['CARE'],
            ['NORTH'],
            ['CARE'],
            ['WEST'],
            ['COLLECT_FERTILIZER']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['NORTH'],
            ['FEED'],
            ['WATER'],
            ['WATER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['NORTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['WATER'],
            ['FEED'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['FERTILIZE'],
            ['FERTILIZE'],
            ['NORTH'],
            ['NORTH'],
            ['WATER'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['EAST'],
            ['WATER'],
            ['WATER'],
            ['SOUTH'],
            ['HARVEST'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['FEED'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['WATER'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['FERTILIZE'],
            ['WEST'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['HARVEST'],
            ['WEST'],
            ['SOUTH'],
            ['WATER'],
            ['WEST']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['PASS'],
            ['SOUTH'],
            ['EAST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['PASS'],
            ['SOUTH'],
            ['WATER'],
            ['FERTILIZE']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS'],
            ['EAST'],
            ['NORTH'],
            ['WATER']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['PLANT', 'MELON'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['PASS'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'],
            ['DIG'],
            ['DROP'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['PASS'],
            ['DROP'],
            ['SOUTH'],
            ['EAST']],
  'market': [['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['PLANT', 'STRAWBERRY'],
            ['PASS'],
            ['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['NORTH'],
            ['EAST']],
  'market': [['BUY_SEED', 'STRAWBERRY', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['PASS'],
            ['PLANT', 'STRAWBERRY'],
            ['WEST'],
            ['DROP'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['SOUTH'],
            ['EAST']],
  'market': [['SELL', 'MILK', 3], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['PASS'],
            ['WATER'],
            ['WEST'],
            ['PASS'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['NORTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WOOL', 6],
             ['SELL', 'MILK', 6],
             ['SELL', 'WHEAT', 11],
             ['SELL', 'STRAWBERRY', 2],
             ['SELL', 'FERTILIZER', 6],
             ['BUY_PRODUCT', 'WHEAT', 1],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['NORTH'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 11], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 8], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['CARE'],
            ['HARVEST'],
            ['SOUTH'],
            ['EAST'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 8], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['HARVEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['FEED'],
            ['NORTH'],
            ['WATER'],
            ['CARE'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['EAST'],
            ['WEST'],
            ['CARE'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 2],
  'hands': [['NORTH'],
            ['FEED'],
            ['EAST'],
            ['WATER'],
            ['NORTH'],
            ['HARVEST'],
            ['CARE'],
            ['WATER'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['SOUTH'],
            ['CARE'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['EAST'],
            ['WATER'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['HARVEST'],
            ['NORTH'],
            ['NORTH'],
            ['HARVEST'],
            ['EAST']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'],
            ['FEED'],
            ['EAST'],
            ['WEST'],
            ['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['HARVEST'],
            ['CARE'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['WATER'],
            ['EAST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['FEED'],
            ['EAST'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['WATER'],
            ['WATER'],
            ['WEST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['EAST'],
            ['WATER'],
            ['WATER'],
            ['WEST'],
            ['HARVEST'],
            ['WATER'],
            ['EAST'],
            ['NORTH'],
            ['HARVEST'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['CARE'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['EAST'],
            ['WEST'],
            ['SOUTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['COLLECT_FERTILIZER'],
  'hands': [['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['WATER'],
            ['SOUTH'],
            ['FERTILIZE'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['DROP'],
            ['CARE']],
  'market': [['SELL', 'MILK', 6], ['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 2]]},
 {'farmer': ['EAST'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['WATER'],
            ['FERTILIZE'],
            ['SOUTH'],
            ['EAST'],
            ['PASS'],
            ['COLLECT_FERTILIZER']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['EAST'],
            ['WATER'],
            ['WEST'],
            ['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['EAST'],
            ['WATER'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['WEST']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['DROP'],
            ['WEST'],
            ['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['WEST']],
  'market': [['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['EAST'],
  'hands': [['PASS'],
            ['WEST'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['DROP']],
  'market': [['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['PASS'],
            ['WEST'],
            ['DROP'],
            ['PLANT', 'STRAWBERRY'],
            ['SOUTH'],
            ['DROP'],
            ['EAST'],
            ['PASS'],
            ['DROP'],
            ['PASS'],
            ['PASS']],
  'market': [['SELL', 'WOOL', 10], ['SELL', 'STRAWBERRY', 4], ['SELL', 'FERTILIZER', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['PASS'],
            ['DROP'],
            ['PASS'],
            ['WATER'],
            ['EAST'],
            ['PASS'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS']],
  'market': [['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WOOL', 6],
             ['SELL', 'STRAWBERRY', 4],
             ['SELL', 'WHEAT', 13],
             ['SELL', 'FERTILIZER', 3],
             ['HIRE'],
             ['HIRE'],
             ['HIRE'],
             ['BUY_ANIMAL', 'SHEEP', 3]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['BUILD_PASTURE'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 17], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['WEST']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['BUILD_PASTURE'],
            ['CARE'],
            ['WEST'],
            ['PICKUP', 'SHEEP', 2],
            ['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['PICKUP', 'SHEEP', 1],
            ['CARE'],
            ['WEST'],
            ['SOUTH'],
            ['EAST'],
            ['HARVEST'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['CARE'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WATER'],
            ['SOUTH'],
            ['CARE'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['NORTH'],
            ['SOUTH'],
            ['FEED'],
            ['SOUTH'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['CARE'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['PLACE', 'SHEEP'],
            ['HARVEST'],
            ['WATER'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 2],
  'hands': [['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['CARE'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH'],
            ['CARE'],
            ['NORTH'],
            ['WEST'],
            ['WEST']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['FEED'],
            ['WATER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['CARE'],
            ['CARE'],
            ['NORTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['WEST'],
            ['WATER'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['WATER'],
            ['NORTH'],
            ['NORTH'],
            ['WATER'],
            ['WEST'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['NORTH'],
            ['FEED'],
            ['EAST'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['WATER'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['EAST'],
            ['WEST'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['FEED'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['WATER'],
            ['WATER'],
            ['WEST']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['WEST'],
            ['WATER'],
            ['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH'],
            ['SOUTH'],
            ['WEST'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['CARE'],
  'hands': [['WEST'],
            ['WATER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['NORTH'],
            ['EAST'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['WEST'],
            ['WATER'],
            ['WEST'],
            ['EAST'],
            ['HARVEST'],
            ['FEED'],
            ['WEST'],
            ['HARVEST'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['WEST'],
            ['PLACE', 'SHEEP'],
            ['EAST'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WATER'],
            ['NORTH'],
            ['SOUTH'],
            ['HARVEST'],
            ['CARE'],
            ['EAST'],
            ['FEED'],
            ['SOUTH'],
            ['WATER'],
            ['EAST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['FERTILIZE'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['DROP'],
            ['FEED'],
            ['DROP'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST']],
  'market': [['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['EAST'],
            ['WEST'],
            ['PASS'],
            ['SOUTH'],
            ['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['DROP']],
  'market': [['SELL', 'STRAWBERRY', 2], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['PASS'],
            ['EAST'],
            ['PASS'],
            ['EAST'],
            ['SOUTH'],
            ['PASS']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'MILK', 9],
             ['SELL', 'STRAWBERRY', 4],
             ['SELL', 'WHEAT', 14],
             ['SELL', 'FERTILIZER', 8],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['HARVEST'], ['CARE'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['SOUTH']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['PICKUP', 'SHEEP', 1],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['NORTH'],
            ['EAST'],
            ['CARE'],
            ['NORTH'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['WEST'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['CARE'],
            ['CARE'],
            ['EAST'],
            ['FEED'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['CARE'],
            ['CARE']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['EAST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['NORTH'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['EAST'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 2],
  'hands': [['EAST'],
            ['CARE'],
            ['EAST'],
            ['WEST'],
            ['FEED'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['NORTH'],
            ['EAST'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['EAST'],
            ['CARE'],
            ['CARE']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['NORTH'],
            ['FEED'],
            ['WEST'],
            ['WATER'],
            ['NORTH'],
            ['NORTH'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['WATER'],
            ['NORTH'],
            ['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['HARVEST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['WEST'],
            ['NORTH'],
            ['WATER'],
            ['FEED'],
            ['WEST'],
            ['WATER'],
            ['NORTH'],
            ['EAST'],
            ['SOUTH'],
            ['WATER']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['HARVEST'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['WATER'],
            ['EAST'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['EAST'],
            ['WATER'],
            ['NORTH'],
            ['EAST'],
            ['PICKUP', 'WHEAT', 4],
            ['WATER']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['EAST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['CARE'],
  'hands': [['WEST'],
            ['FERTILIZE'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['FEED'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['COLLECT_FERTILIZER'],
  'hands': [['NORTH'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['PLACE', 'SHEEP'],
            ['WATER'],
            ['SOUTH'],
            ['HARVEST'],
            ['WATER'],
            ['EAST'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['CARE'],
            ['SOUTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['WEST'],
            ['FEED'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'],
            ['FERTILIZE'],
            ['FERTILIZE'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['WATER'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['HARVEST'],
            ['SOUTH'],
            ['FEED'],
            ['WEST']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['WATER'],
            ['WATER'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['DROP'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['WEST']],
  'market': [['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['DROP'],
  'hands': [['DROP'],
            ['WEST'],
            ['SOUTH'],
            ['PASS'],
            ['PASS'],
            ['NORTH'],
            ['EAST'],
            ['SOUTH'],
            ['WEST'],
            ['FEED'],
            ['DROP']],
  'market': [['SELL', 'WOOL', 4],
             ['SELL', 'WHEAT', 12],
             ['SELL', 'FERTILIZER', 4],
             ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'STRAWBERRY', 10],
             ['SELL', 'MILK', 3],
             ['SELL', 'FERTILIZER', 5],
             ['SELL', 'WHEAT', 1],
             ['BUY_PRODUCT', 'WHEAT', 13],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['CARE'], ['NORTH']],
  'market': [['SELL', 'WHEAT', 9], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['WEST']],
  'market': [['SELL', 'WHEAT', 1], ['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['WEST'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH'],
            ['PICKUP', 'FERTILIZER', 3],
            ['CARE'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 2], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['CARE'],
            ['CARE'],
            ['HARVEST'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['HARVEST'],
            ['NORTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['CARE'],
            ['WEST'],
            ['HARVEST'],
            ['FEED'],
            ['WATER'],
            ['CARE'],
            ['WATER'],
            ['CARE'],
            ['CARE'],
            ['COLLECT_FERTILIZER']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['FEED'],
            ['NORTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WATER'],
            ['NORTH'],
            ['WATER'],
            ['EAST'],
            ['WEST'],
            ['CARE'],
            ['NORTH'],
            ['CARE']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'],
            ['SOUTH'],
            ['FEED'],
            ['EAST'],
            ['FEED'],
            ['WEST'],
            ['WATER'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['HARVEST'],
            ['SOUTH'],
            ['CARE'],
            ['EAST'],
            ['CARE'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['WEST'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['WATER'],
            ['WATER']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['WATER'],
            ['EAST'],
            ['EAST'],
            ['NORTH'],
            ['WATER'],
            ['FERTILIZE'],
            ['FERTILIZE'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['NORTH'],
            ['FEED'],
            ['EAST'],
            ['FEED'],
            ['WATER'],
            ['WEST'],
            ['NORTH'],
            ['WATER'],
            ['SOUTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['WEST'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['EAST'],
            ['WEST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['WATER'],
            ['FERTILIZE']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['NORTH'],
            ['FERTILIZE'],
            ['EAST'],
            ['WATER'],
            ['FEED'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['WEST'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['WEST'],
            ['FEED'],
            ['SOUTH'],
            ['EAST'],
            ['HARVEST'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['FERTILIZE'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['NORTH'],
            ['WEST'],
            ['HARVEST'],
            ['NORTH'],
            ['WEST'],
            ['FERTILIZE']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['HARVEST'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['WATER']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['WATER'],
            ['FERTILIZE'],
            ['DROP'],
            ['WATER'],
            ['WEST'],
            ['HARVEST'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'],
            ['SOUTH'],
            ['PASS'],
            ['SOUTH'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'],
            ['EAST'],
            ['PASS'],
            ['WEST'],
            ['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['EAST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'MILK', 15],
             ['SELL', 'WOOL', 12],
             ['SELL', 'STRAWBERRY', 10],
             ['SELL', 'WHEAT', 14],
             ['SELL', 'FERTILIZER', 7],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['CARE'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['SOUTH']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH'],
            ['CARE'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['CARE'],
            ['NORTH'],
            ['WEST'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['NORTH'],
            ['FEED'],
            ['EAST'],
            ['FEED'],
            ['WATER'],
            ['EAST'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WEST'],
            ['CARE']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['CARE'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['CARE'],
            ['WEST'],
            ['CARE'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['CARE'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['EAST'],
            ['COLLECT_FERTILIZER'],
            ['FEED'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['NORTH'],
            ['CARE'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['NORTH'],
            ['EAST'],
            ['WATER'],
            ['FEED'],
            ['WATER'],
            ['EAST'],
            ['WATER'],
            ['WATER'],
            ['HARVEST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WATER'],
            ['WEST'],
            ['FEED'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['WATER'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['FERTILIZE'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['FERTILIZE'],
            ['WATER'],
            ['HARVEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['WEST'],
            ['EAST'],
            ['WATER'],
            ['FEED'],
            ['WATER'],
            ['WATER'],
            ['NORTH'],
            ['SOUTH'],
            ['CARE'],
            ['WATER']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['NORTH'],
            ['WATER'],
            ['EAST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['NORTH'],
            ['HARVEST'],
            ['WEST'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['FERTILIZE'],
            ['SOUTH'],
            ['CARE'],
            ['HARVEST'],
            ['SOUTH'],
            ['EAST'],
            ['HARVEST'],
            ['NORTH'],
            ['WEST'],
            ['WATER'],
            ['FERTILIZE']],
  'market': []},
 {'farmer': ['COLLECT_FERTILIZER'],
  'hands': [['WATER'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['EAST'],
            ['WATER'],
            ['HARVEST'],
            ['NORTH'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['FEED'],
            ['WATER'],
            ['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['HARVEST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['NORTH'],
            ['WATER'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['WEST']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['EAST'],
            ['NORTH'],
            ['EAST'],
            ['HARVEST']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['EAST'],
            ['WEST'],
            ['WEST'],
            ['FERTILIZE'],
            ['WATER'],
            ['SOUTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['DROP'],
  'hands': [['WEST'],
            ['WATER'],
            ['WEST'],
            ['DROP'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH']],
  'market': [['SELL', 'MILK', 6],
             ['SELL', 'WHEAT', 12],
             ['SELL', 'STRAWBERRY', 2],
             ['SELL', 'FERTILIZER', 2],
             ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['PASS'],
            ['EAST'],
            ['NORTH'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['DROP'],
            ['EAST']],
  'market': [['SELL', 'MILK', 9],
             ['SELL', 'STRAWBERRY', 2],
             ['SELL', 'WHEAT', 1],
             ['BUY_PRODUCT', 'WHEAT', 12]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'STRAWBERRY', 12],
             ['SELL', 'WHEAT', 13],
             ['SELL', 'FERTILIZER', 7],
             ['BUY_PRODUCT', 'WHEAT', 1],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['HARVEST'], ['CARE'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 13], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['PICKUP', 'FERTILIZER', 3],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['CARE'],
            ['PICKUP', 'WHEAT', 4],
            ['COLLECT_FERTILIZER'],
            ['PICKUP', 'FERTILIZER', 3],
            ['CARE'],
            ['NORTH'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['CARE'],
            ['SOUTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 1],
  'hands': [['CARE'],
            ['WEST'],
            ['NORTH'],
            ['HARVEST'],
            ['WEST'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['NORTH'],
            ['WEST'],
            ['FEED'],
            ['EAST'],
            ['FEED'],
            ['HARVEST'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['WATER'],
            ['WEST']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['FERTILIZE'],
            ['CARE'],
            ['NORTH'],
            ['EAST'],
            ['WEST'],
            ['CARE'],
            ['SOUTH'],
            ['CARE']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['WATER'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['WATER'],
            ['FEED'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['EAST'],
            ['WEST'],
            ['WATER']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['FERTILIZE'],
            ['WATER'],
            ['WATER'],
            ['FERTILIZE'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['WATER'],
            ['EAST'],
            ['EAST'],
            ['FEED'],
            ['WATER'],
            ['WEST'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['HARVEST']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['NORTH'],
            ['EAST'],
            ['WATER'],
            ['CARE'],
            ['WEST'],
            ['HARVEST'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['DIG'],
            ['FERTILIZE'],
            ['FEED'],
            ['EAST'],
            ['COLLECT_FERTILIZER'],
            ['FERTILIZE'],
            ['WEST'],
            ['WATER'],
            ['FERTILIZE'],
            ['NORTH'],
            ['WATER']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['HARVEST'],
            ['NORTH'],
            ['WATER'],
            ['NORTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['HARVEST'],
            ['NORTH'],
            ['EAST'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['WATER']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['HARVEST'],
            ['FEED'],
            ['WATER'],
            ['FEED'],
            ['WATER'],
            ['WEST'],
            ['NORTH'],
            ['HARVEST'],
            ['NORTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['CARE'],
            ['HARVEST'],
            ['CARE'],
            ['HARVEST'],
            ['FERTILIZE'],
            ['NORTH'],
            ['EAST'],
            ['HARVEST'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['NORTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['PLANT', 'STRAWBERRY'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['WATER'],
            ['EAST'],
            ['WATER'],
            ['EAST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['EAST'],
            ['EAST'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['HARVEST'],
            ['EAST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['EAST'],
            ['FERTILIZE'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'],
            ['EAST'],
            ['WATER'],
            ['HARVEST'],
            ['EAST'],
            ['SOUTH'],
            ['HARVEST'],
            ['EAST'],
            ['SOUTH'],
            ['EAST'],
            ['WATER']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['EAST'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['DROP'],
            ['EAST'],
            ['SOUTH'],
            ['PASS'],
            ['WATER'],
            ['EAST'],
            ['SOUTH']],
  'market': [['SELL', 'MILK', 3], ['SELL', 'FERTILIZER', 1], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'MELON', 24],
             ['SELL', 'STRAWBERRY', 16],
             ['SELL', 'MILK', 9],
             ['SELL', 'WOOL', 4],
             ['SELL', 'WHEAT', 14],
             ['HIRE'],
             ['HIRE'],
             ['HIRE'],
             ['SELL', 'FERTILIZER', 5]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['CARE'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['SOUTH']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['EAST'],
            ['PICKUP', 'FERTILIZER', 3],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['HARVEST'],
            ['PICKUP', 'FERTILIZER', 3],
            ['CARE'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['EAST'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['PICKUP', 'FERTILIZER', 3],
            ['NORTH'],
            ['CARE'],
            ['WEST'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['CARE'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['WEST'],
            ['FEED'],
            ['EAST'],
            ['FEED'],
            ['WEST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['CARE'],
            ['CARE'],
            ['NORTH'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['CARE'],
            ['SOUTH'],
            ['COLLECT_FERTILIZER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'],
            ['WEST'],
            ['FEED'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['HARVEST'],
            ['HARVEST'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['FERTILIZE'],
            ['CARE'],
            ['CARE'],
            ['HARVEST'],
            ['FEED'],
            ['WATER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['FERTILIZE'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['HARVEST'],
            ['EAST'],
            ['EAST'],
            ['CARE'],
            ['WEST'],
            ['WATER'],
            ['NORTH'],
            ['NORTH'],
            ['WATER'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['NORTH'],
            ['NORTH'],
            ['FEED'],
            ['WATER'],
            ['HARVEST'],
            ['WATER'],
            ['FERTILIZE'],
            ['HARVEST'],
            ['HARVEST'],
            ['SOUTH'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WATER'],
            ['WEST'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['EAST'],
            ['HARVEST'],
            ['FERTILIZE'],
            ['EAST'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['WATER'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['WEST'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['FEED'],
            ['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['WATER'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['HARVEST'],
            ['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['HARVEST'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['SOUTH'],
            ['FEED'],
            ['NORTH'],
            ['EAST'],
            ['WATER'],
            ['EAST'],
            ['DROP'],
            ['HARVEST'],
            ['NORTH'],
            ['WEST']],
  'market': [['SELL', 'WOOL', 4], ['SELL', 'STRAWBERRY', 4]]},
 {'farmer': ['NORTH'],
  'hands': [['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['FEED'],
            ['NORTH'],
            ['HARVEST'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['HARVEST'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['DROP'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['DROP']],
  'market': [['SELL', 'STRAWBERRY', 4],
             ['SELL', 'MELON', 6],
             ['SELL', 'MILK', 3],
             ['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['EAST'],
            ['WEST'],
            ['WEST'],
            ['PASS'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['PASS']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['EAST'],
            ['HARVEST'],
            ['WEST'],
            ['PASS'],
            ['NORTH'],
            ['WEST'],
            ['HARVEST'],
            ['WEST'],
            ['NORTH'],
            ['PASS']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'STRAWBERRY', 18],
             ['SELL', 'WOOL', 8],
             ['SELL', 'MILK', 6],
             ['SELL', 'WHEAT', 14],
             ['SELL', 'FERTILIZER', 9],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['HARVEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['CARE'],
            ['WEST'],
            ['WEST'],
            ['PICKUP', 'WHEAT', 4],
            ['COLLECT_FERTILIZER']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['WEST'],
            ['WEST'],
            ['PICKUP', 'WHEAT', 4],
            ['SOUTH'],
            ['PICKUP', 'FERTILIZER', 3],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['PICKUP', 'FERTILIZER', 3],
            ['CARE'],
            ['PICKUP', 'FERTILIZER', 3],
            ['SOUTH'],
            ['CARE'],
            ['HARVEST'],
            ['NORTH'],
            ['CARE'],
            ['EAST'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['PICKUP', 'FERTILIZER', 1],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['CARE'],
            ['WEST'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['FEED'],
            ['CARE'],
            ['NORTH'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['EAST']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['HARVEST'],
            ['WEST'],
            ['FEED'],
            ['SOUTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WATER'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'],
            ['HARVEST'],
            ['CARE'],
            ['WEST'],
            ['NORTH'],
            ['CARE'],
            ['EAST'],
            ['NORTH'],
            ['CARE'],
            ['SOUTH'],
            ['CARE']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['COLLECT_FERTILIZER'],
            ['WEST'],
            ['HARVEST'],
            ['WEST'],
            ['FEED'],
            ['NORTH'],
            ['WATER'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['COLLECT_FERTILIZER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['EAST'],
            ['WATER'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['WEST'],
            ['WATER'],
            ['NORTH'],
            ['WATER'],
            ['FEED'],
            ['NORTH'],
            ['WATER'],
            ['HARVEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['SOUTH'],
            ['FEED'],
            ['WEST'],
            ['SOUTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['WATER'],
            ['CARE'],
            ['WATER'],
            ['SOUTH'],
            ['EAST'],
            ['WATER'],
            ['WATER'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['EAST'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['FERTILIZE'],
            ['HARVEST'],
            ['NORTH'],
            ['FERTILIZE']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['NORTH'],
            ['WATER'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['EAST'],
            ['EAST'],
            ['WEST'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WATER'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['WEST'],
            ['HARVEST'],
            ['WATER'],
            ['NORTH'],
            ['WATER'],
            ['NORTH']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['HARVEST'],
            ['FERTILIZE'],
            ['FEED'],
            ['NORTH'],
            ['SOUTH'],
            ['WEST'],
            ['NORTH'],
            ['EAST'],
            ['FERTILIZE'],
            ['NORTH'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['CARE'],
            ['NORTH'],
            ['FEED'],
            ['WEST'],
            ['EAST'],
            ['WATER'],
            ['EAST'],
            ['HARVEST'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['FERTILIZE'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['EAST'],
            ['HARVEST'],
            ['SOUTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['DIG'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['NORTH'],
            ['WATER']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['NORTH'],
            ['DROP'],
            ['DIG'],
            ['FERTILIZE'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['FERTILIZE'],
            ['EAST'],
            ['HARVEST']],
  'market': [['SELL', 'MILK', 3]]},
 {'farmer': ['NORTH'],
  'hands': [['EAST'],
            ['DIG'],
            ['PASS'],
            ['NORTH'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['DIG'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'STRAWBERRY', 20],
             ['SELL', 'WOOL', 11],
             ['SELL', 'MILK', 9],
             ['SELL', 'MELON', 11],
             ['SELL', 'WHEAT', 14],
             ['HIRE'],
             ['HIRE'],
             ['HIRE'],
             ['SELL', 'FERTILIZER', 7]]},
 {'farmer': ['FEED'],
  'hands': [['HARVEST'], ['CARE'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['WEST'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['PICKUP', 'FERTILIZER', 3],
            ['NORTH'],
            ['NORTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['PICKUP', 'WHEAT', 4],
            ['HARVEST'],
            ['CARE'],
            ['CARE'],
            ['PICKUP', 'FERTILIZER', 3],
            ['CARE'],
            ['EAST'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['EAST'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 1],
  'hands': [['FEED'],
            ['SOUTH'],
            ['EAST'],
            ['NORTH'],
            ['FEED'],
            ['CARE'],
            ['CARE'],
            ['CARE'],
            ['SOUTH'],
            ['SOUTH'],
            ['CARE']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['COLLECT_FERTILIZER'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['DIG'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'],
            ['WATER'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['CARE'],
            ['HARVEST'],
            ['CARE'],
            ['HARVEST'],
            ['SOUTH'],
            ['WATER']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['FEED'],
            ['WEST'],
            ['EAST'],
            ['EAST'],
            ['FEED'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['HARVEST'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['FERTILIZE'],
            ['WATER'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['HARVEST'],
            ['NORTH'],
            ['WATER'],
            ['SOUTH'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['EAST'],
  'hands': [['FEED'],
            ['WATER'],
            ['NORTH'],
            ['EAST'],
            ['SOUTH'],
            ['HARVEST'],
            ['NORTH'],
            ['DIG'],
            ['WEST'],
            ['NORTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['WATER'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['CARE'],
  'hands': [['SOUTH'],
            ['SOUTH'],
            ['HARVEST'],
            ['FERTILIZE'],
            ['FEED'],
            ['FERTILIZE'],
            ['NORTH'],
            ['DIG'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['HARVEST'],
  'hands': [['EAST'],
            ['WATER'],
            ['NORTH'],
            ['NORTH'],
            ['CARE'],
            ['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['SOUTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['COLLECT_FERTILIZER'],
  'hands': [['FEED'],
            ['WEST'],
            ['FERTILIZE'],
            ['HARVEST'],
            ['HARVEST'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['HARVEST']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['NORTH'],
            ['WATER'],
            ['SOUTH'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['WEST'],
            ['WATER'],
            ['WATER'],
            ['WEST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['NORTH'],
            ['SOUTH'],
            ['WEST'],
            ['HARVEST'],
            ['SOUTH'],
            ['FERTILIZE'],
            ['HARVEST'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['WEST']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['HARVEST'],
            ['WATER'],
            ['HARVEST'],
            ['SOUTH'],
            ['EAST'],
            ['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['EAST'],
            ['HARVEST'],
            ['HARVEST']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['FEED'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['FERTILIZE'],
            ['NORTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['HARVEST'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['FEED'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['DIG'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['HARVEST'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['DIG'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'STRAWBERRY', 25],
             ['SELL', 'WOOL', 10],
             ['SELL', 'MILK', 9],
             ['SELL', 'MELON', 6],
             ['SELL', 'WHEAT', 14],
             ['HIRE'],
             ['HIRE'],
             ['HIRE'],
             ['SELL', 'FERTILIZER', 8]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['CARE'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['WEST']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['WEST'],
            ['PICKUP', 'WHEAT', 4],
            ['WEST'],
            ['PICKUP', 'FERTILIZER', 3],
            ['CARE'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['PICKUP', 'FERTILIZER', 3],
            ['PICKUP', 'FERTILIZER', 3],
            ['PICKUP', 'FERTILIZER', 1],
            ['WEST'],
            ['EAST'],
            ['CARE'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['NORTH'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WEST'],
            ['HARVEST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['CARE'],
            ['WEST'],
            ['NORTH'],
            ['FEED'],
            ['WATER'],
            ['CARE'],
            ['CARE'],
            ['NORTH'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['CARE'],
            ['HARVEST'],
            ['FEED'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['HARVEST'],
            ['NORTH'],
            ['NORTH'],
            ['HARVEST'],
            ['NORTH'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['FEED'],
            ['SOUTH'],
            ['CARE'],
            ['HARVEST'],
            ['EAST'],
            ['CARE'],
            ['CARE']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['FEED'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WATER'],
            ['EAST'],
            ['COLLECT_FERTILIZER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['FERTILIZE'],
            ['WEST'],
            ['CARE'],
            ['NORTH'],
            ['FEED'],
            ['WEST'],
            ['EAST'],
            ['WATER'],
            ['EAST'],
            ['NORTH'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['COLLECT_FERTILIZER'],
            ['FERTILIZE'],
            ['SOUTH'],
            ['WATER'],
            ['WATER'],
            ['WEST'],
            ['WATER'],
            ['HARVEST'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['FERTILIZE'],
            ['HARVEST'],
            ['SOUTH'],
            ['WEST'],
            ['SOUTH'],
            ['HARVEST'],
            ['FERTILIZE'],
            ['WATER'],
            ['EAST'],
            ['NORTH'],
            ['FERTILIZE']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['EAST'],
            ['SOUTH'],
            ['WEST'],
            ['FERTILIZE'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['WATER'],
            ['HARVEST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['WATER'],
            ['FEED'],
            ['NORTH'],
            ['EAST'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['EAST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['WEST'],
            ['CARE'],
            ['WEST'],
            ['FEED'],
            ['HARVEST'],
            ['HARVEST'],
            ['WATER'],
            ['NORTH'],
            ['WATER'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['NORTH'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['FERTILIZE'],
            ['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['HARVEST'],
            ['HARVEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['SOUTH'],
            ['WATER'],
            ['HARVEST'],
            ['EAST'],
            ['HARVEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['WEST'],
            ['WATER'],
            ['WEST'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'],
            ['SOUTH'],
            ['FEED'],
            ['WEST'],
            ['SOUTH'],
            ['HARVEST'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['FERTILIZE'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['NORTH'],
            ['WEST'],
            ['WATER'],
            ['WEST'],
            ['NORTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['WEST']],
  'market': []},
 {'farmer': ['WATER'],
  'hands': [['WEST'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['DROP'],
            ['SOUTH']],
  'market': [['SELL', 'STRAWBERRY', 6], ['SELL', 'MILK', 3]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['NORTH'],
            ['WATER'],
            ['FERTILIZE'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['PASS'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'STRAWBERRY', 22],
             ['SELL', 'WOOL', 12],
             ['SELL', 'MILK', 9],
             ['SELL', 'MELON', 6],
             ['SELL', 'WHEAT', 14],
             ['HIRE'],
             ['HIRE'],
             ['HIRE'],
             ['SELL', 'FERTILIZER', 5]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['HARVEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['CARE'],
            ['WEST'],
            ['WEST'],
            ['PICKUP', 'WHEAT', 4],
            ['COLLECT_FERTILIZER']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['SOUTH'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH'],
            ['PICKUP', 'FERTILIZER', 3],
            ['WEST'],
            ['NORTH'],
            ['EAST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['SOUTH'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['CARE'],
            ['NORTH'],
            ['CARE'],
            ['SOUTH'],
            ['NORTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['WEST'],
            ['NORTH'],
            ['FEED'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['NORTH'],
            ['WATER'],
            ['FEED'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['CARE'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['CARE'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['CARE'],
            ['HARVEST'],
            ['HARVEST'],
            ['CARE']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['CARE'],
            ['FEED'],
            ['NORTH'],
            ['WATER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['FERTILIZE'],
            ['WATER'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['HARVEST'],
            ['HARVEST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['EAST'],
            ['HARVEST'],
            ['FEED'],
            ['WEST'],
            ['FEED'],
            ['WATER'],
            ['WATER'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['FERTILIZE'],
            ['SOUTH'],
            ['CARE'],
            ['WEST'],
            ['SOUTH'],
            ['FERTILIZE'],
            ['EAST'],
            ['NORTH'],
            ['WATER'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['SOUTH'],
  'hands': [['NORTH'],
            ['WATER'],
            ['HARVEST'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['HARVEST'],
            ['HARVEST'],
            ['EAST'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['EAST'],
            ['WATER'],
            ['EAST'],
            ['EAST'],
            ['WEST'],
            ['HARVEST'],
            ['EAST']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['DIG'],
            ['WATER'],
            ['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['WATER'],
            ['WEST'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['NORTH'],
            ['WEST'],
            ['HARVEST'],
            ['WEST'],
            ['FERTILIZE'],
            ['WATER'],
            ['HARVEST'],
            ['HARVEST'],
            ['SOUTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['WATER'],
            ['WEST'],
            ['FEED'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['WEST'],
            ['CARE'],
            ['HARVEST'],
            ['FEED'],
            ['NORTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['HARVEST'],
            ['HARVEST'],
            ['HARVEST'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['NORTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['HARVEST'],
            ['WEST']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['HARVEST'],
            ['NORTH'],
            ['SOUTH'],
            ['DROP']],
  'market': [['SELL', 'STRAWBERRY', 6], ['SELL', 'WOOL', 4]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['FERTILIZE'],
            ['EAST'],
            ['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['PASS']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['HARVEST'],
            ['EAST'],
            ['WEST'],
            ['SOUTH'],
            ['WATER'],
            ['PASS'],
            ['WEST'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['PASS']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS'],
            ['WEST'],
            ['WEST'],
            ['EAST'],
            ['EAST'],
            ['PASS']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'STRAWBERRY', 36],
             ['SELL', 'MILK', 9],
             ['SELL', 'WOOL', 4],
             ['SELL', 'WHEAT', 14],
             ['SELL', 'FERTILIZER', 9],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['HARVEST'], ['CARE'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['WEST'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['PICKUP', 'WHEAT', 4],
            ['PICKUP', 'FERTILIZER', 3],
            ['NORTH'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['PICKUP', 'WHEAT', 4],
            ['HARVEST'],
            ['CARE'],
            ['PICKUP', 'FERTILIZER', 3],
            ['CARE'],
            ['HARVEST'],
            ['EAST'],
            ['CARE'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['CARE'],
            ['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 1],
  'hands': [['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['CARE'],
            ['FEED'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['CARE']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['FEED'],
            ['SOUTH'],
            ['HARVEST'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['NORTH'],
            ['WATER'],
            ['HARVEST'],
            ['CARE'],
            ['COLLECT_FERTILIZER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'],
            ['FERTILIZE'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['CARE'],
            ['WEST'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['NORTH'],
            ['WEST'],
            ['WATER'],
            ['CARE'],
            ['FEED'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['HARVEST'],
            ['NORTH'],
            ['COLLECT_FERTILIZER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['FEED'],
            ['FERTILIZE'],
            ['EAST'],
            ['NORTH'],
            ['EAST'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['WATER'],
            ['FEED'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['HARVEST'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['EAST'],
            ['WEST'],
            ['SOUTH'],
            ['NORTH'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['FEED'],
            ['NORTH'],
            ['WATER'],
            ['HARVEST'],
            ['SOUTH'],
            ['DROP'],
            ['WATER'],
            ['WATER'],
            ['WATER'],
            ['HARVEST'],
            ['NORTH']],
  'market': [['SELL', 'MILK', 6], ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'],
            ['NORTH'],
            ['FERTILIZE'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['HARVEST'],
            ['NORTH'],
            ['EAST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['EAST'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['EAST'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['EAST'],
            ['EAST'],
            ['WATER'],
            ['HARVEST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['EAST'],
            ['NORTH'],
            ['FERTILIZE'],
            ['HARVEST'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['NORTH'],
  'hands': [['FEED'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH'],
            ['FEED'],
            ['EAST'],
            ['NORTH'],
            ['EAST'],
            ['EAST'],
            ['WATER'],
            ['WEST']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['SOUTH'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['HARVEST'],
            ['EAST'],
            ['WEST'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['HARVEST'],
            ['DIG'],
            ['NORTH'],
            ['EAST'],
            ['WEST'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['HARVEST'],
  'hands': [['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['EAST'],
            ['DIG'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['FERTILIZE'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['DROP'],
            ['SOUTH'],
            ['SOUTH']],
  'market': [['SELL', 'STRAWBERRY', 4]]},
 {'farmer': ['DIG'],
  'hands': [['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['DROP'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['DROP'],
            ['PASS'],
            ['SOUTH'],
            ['SOUTH']],
  'market': [['SELL', 'STRAWBERRY', 8], ['SELL', 'MILK', 3], ['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'WOOL', 8],
             ['SELL', 'STRAWBERRY', 12],
             ['SELL', 'WHEAT', 14],
             ['SELL', 'MILK', 3],
             ['SELL', 'FERTILIZER', 8],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['CARE'], ['CARE'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['WEST'],
            ['PICKUP', 'WHEAT', 4],
            ['EAST'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['HARVEST'],
            ['CARE'],
            ['NORTH'],
            ['HARVEST'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['CARE'],
            ['COLLECT_FERTILIZER'],
            ['FEED'],
            ['CARE'],
            ['FEED'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['NORTH'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['CARE'],
            ['NORTH'],
            ['NORTH'],
            ['WATER'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['CARE'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['CARE'],
            ['NORTH'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['CARE'],
            ['WEST'],
            ['NORTH']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['EAST'],
            ['NORTH'],
            ['FEED'],
            ['HARVEST'],
            ['FEED'],
            ['CARE'],
            ['NORTH'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['HARVEST'],
            ['CARE'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['NORTH'],
            ['EAST'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['NORTH'],
  'hands': [['EAST'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['HARVEST'],
            ['WATER'],
            ['HARVEST'],
            ['HARVEST'],
            ['WATER'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['HARVEST'],
            ['NORTH'],
            ['EAST'],
            ['EAST'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['HARVEST']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['EAST'],
            ['NORTH'],
            ['EAST'],
            ['EAST'],
            ['FEED'],
            ['WEST'],
            ['NORTH'],
            ['WATER'],
            ['EAST'],
            ['WATER'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['HARVEST'],
            ['NORTH'],
            ['FEED'],
            ['EAST'],
            ['CARE'],
            ['FERTILIZE'],
            ['EAST'],
            ['HARVEST'],
            ['HARVEST'],
            ['WEST'],
            ['WATER']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'],
            ['WEST'],
            ['SOUTH'],
            ['HARVEST'],
            ['HARVEST'],
            ['WEST'],
            ['DIG'],
            ['NORTH'],
            ['NORTH'],
            ['WATER'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['HARVEST'],
            ['EAST'],
            ['SOUTH'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['EAST'],
            ['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['EAST'],
            ['FEED'],
            ['SOUTH'],
            ['SOUTH'],
            ['HARVEST'],
            ['EAST'],
            ['HARVEST'],
            ['WEST'],
            ['EAST'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['EAST'],
            ['WEST'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['DIG'],
            ['NORTH'],
            ['WEST'],
            ['WATER'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['DIG'],
            ['WEST'],
            ['WEST'],
            ['FEED'],
            ['SOUTH'],
            ['SOUTH'],
            ['WATER'],
            ['WEST'],
            ['NORTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['FEED'],
  'hands': [['DROP'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['WATER'],
            ['SOUTH']],
  'market': [['SELL', 'WOOL', 4],
             ['SELL', 'STRAWBERRY', 6],
             ['SELL', 'FERTILIZER', 1],
             ['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['DROP'],
            ['WATER'],
            ['HARVEST'],
            ['WEST'],
            ['HARVEST'],
            ['SOUTH'],
            ['EAST'],
            ['EAST']],
  'market': [['SELL', 'WOOL', 4],
             ['SELL', 'MILK', 3],
             ['SELL', 'STRAWBERRY', 2],
             ['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS'],
            ['SOUTH'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['EAST'],
            ['EAST']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['PASS'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS'],
            ['FERTILIZE'],
            ['EAST'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['EAST'],
            ['EAST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [],
  'market': [['SELL', 'STRAWBERRY', 26],
             ['SELL', 'MILK', 6],
             ['SELL', 'WOOL', 4],
             ['SELL', 'WHEAT', 14],
             ['SELL', 'FERTILIZER', 10],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['FEED'],
  'hands': [['COLLECT_FERTILIZER'], ['HARVEST'], ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 14], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['WEST'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['WEST'],
            ['PICKUP', 'WHEAT', 4],
            ['NORTH']],
  'market': [['SELL', 'WHEAT', 10], ['BUY_PRODUCT', 'WHEAT', 1], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['PICKUP', 'WHEAT', 4],
            ['SOUTH'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['EAST'],
            ['WEST'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 10], ['HIRE'], ['HIRE']]},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['NORTH'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['WEST']],
  'market': []},
 {'farmer': ['PICKUP', 'WHEAT', 4],
  'hands': [['FEED'],
            ['WEST'],
            ['HARVEST'],
            ['WEST'],
            ['FEED'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['NORTH'],
            ['NORTH'],
            ['WEST']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['COLLECT_FERTILIZER'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['WATER'],
            ['HARVEST'],
            ['NORTH'],
            ['NORTH'],
            ['SOUTH'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['WEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['COLLECT_FERTILIZER'],
            ['COLLECT_FERTILIZER'],
            ['DIG'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['WEST']],
  'market': []},
 {'farmer': ['NORTH'],
  'hands': [['FEED'],
            ['HARVEST'],
            ['DIG'],
            ['WATER'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['WEST'],
            ['HARVEST'],
            ['WEST'],
            ['HARVEST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['FEED'],
            ['DIG'],
            ['HARVEST'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['EAST'],
  'hands': [['FEED'],
            ['WATER'],
            ['DIG'],
            ['HARVEST'],
            ['COLLECT_FERTILIZER'],
            ['WEST'],
            ['NORTH'],
            ['FERTILIZE'],
            ['SOUTH'],
            ['DIG'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['SOUTH'],
            ['HARVEST'],
            ['EAST'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['WATER'],
            ['WEST'],
            ['WEST'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['SOUTH'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['DIG'],
            ['WATER'],
            ['WEST'],
            ['HARVEST'],
            ['EAST'],
            ['EAST'],
            ['WEST'],
            ['WEST'],
            ['SOUTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['FEED'],
  'hands': [['EAST'],
            ['EAST'],
            ['EAST'],
            ['HARVEST'],
            ['FEED'],
            ['WEST'],
            ['EAST'],
            ['WATER'],
            ['SOUTH'],
            ['WEST'],
            ['WATER']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['WEST'],
            ['EAST'],
            ['DIG'],
            ['NORTH'],
            ['COLLECT_FERTILIZER'],
            ['SOUTH'],
            ['WATER'],
            ['NORTH'],
            ['WEST'],
            ['WATER'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['FEED'],
  'hands': [['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['WEST'],
            ['FERTILIZE'],
            ['NORTH']],
  'market': []},
 {'farmer': ['EAST'],
  'hands': [['SOUTH'],
            ['EAST'],
            ['SOUTH'],
            ['DIG'],
            ['EAST'],
            ['WATER'],
            ['SOUTH'],
            ['NORTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH']],
  'market': [['BUY_PRODUCT', 'WHEAT', 1]]},
 {'farmer': ['DIG'],
  'hands': [['FEED'],
            ['DROP'],
            ['WEST'],
            ['EAST'],
            ['FEED'],
            ['FERTILIZE'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['NORTH']],
  'market': [['SELL', 'WOOL', 4], ['SELL', 'STRAWBERRY', 6], ['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['PASS'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['EAST'],
            ['EAST']],
  'market': [['BUY_PRODUCT', 'WHEAT', 2]]},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['PASS'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['SOUTH'],
            ['DIG'],
            ['EAST']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['SOUTH'],
            ['PASS'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['DIG'],
            ['SOUTH'],
            ['SOUTH'],
            ['EAST']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['WEST'],
            ['PASS'],
            ['DROP'],
            ['DROP'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['DROP']],
  'market': [['SELL', 'WOOL', 4], ['SELL', 'STRAWBERRY', 6], ['SELL', 'FERTILIZER', 2]]},
 {'farmer': ['PASS'],
  'hands': [['WEST'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['FERTILIZE'],
            ['SOUTH'],
            ['WEST'],
            ['EAST'],
            ['SOUTH'],
            ['SOUTH'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['WEST'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['WATER'],
            ['SOUTH'],
            ['DROP'],
            ['EAST'],
            ['FERTILIZE'],
            ['SOUTH'],
            ['PASS']],
  'market': [['SELL', 'STRAWBERRY', 2], ['SELL', 'FERTILIZER', 1]]},
 {'farmer': ['EAST'],
  'hands': [],
  'market': [['SELL', 'MILK', 12],
             ['SELL', 'WHEAT', 32],
             ['SELL', 'FERTILIZER', 15],
             ['SELL', 'STRAWBERRY', 4],
             ['HIRE'],
             ['HIRE'],
             ['HIRE'],
             ['HIRE']]},
 {'farmer': ['HARVEST'],
  'hands': [['NORTH'], ['WEST'], ['NORTH'], ['WEST']],
  'market': [['HIRE'], ['HIRE'], ['HIRE'], ['HIRE']]},
 {'farmer': ['NORTH'],
  'hands': [['WEST'], ['HARVEST'], ['NORTH'], ['WEST'], ['SOUTH'], ['SOUTH'], ['WEST'], ['SOUTH']],
  'market': [['HIRE'], ['HIRE']]},
 {'farmer': ['EAST'],
  'hands': [['HARVEST'],
            ['SOUTH'],
            ['NORTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['HARVEST'],
  'hands': [['NORTH'],
            ['SOUTH'],
            ['NORTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['SOUTH'],
  'hands': [['WEST'],
            ['HARVEST'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH'],
            ['WEST']],
  'market': []},
 {'farmer': ['WEST'],
  'hands': [['WEST'],
            ['WEST'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['SOUTH']],
  'market': []},
 {'farmer': ['DROP'],
  'hands': [['WEST'],
            ['HARVEST'],
            ['WEST'],
            ['SOUTH'],
            ['HARVEST'],
            ['WEST'],
            ['WEST'],
            ['WEST'],
            ['WEST'],
            ['SOUTH']],
  'market': [['SELL', 'WOOL', 4], ['SELL', 'MILK', 3]]},
 {'farmer': ['PASS'],
  'hands': [['HARVEST'],
            ['SOUTH'],
            ['WEST'],
            ['HARVEST'],
            ['SOUTH'],
            ['HARVEST'],
            ['WEST'],
            ['WEST'],
            ['NORTH'],
            ['NORTH']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['NORTH'],
            ['WEST'],
            ['SOUTH'],
            ['SOUTH'],
            ['NORTH'],
            ['NORTH'],
            ['HARVEST'],
            ['HARVEST'],
            ['NORTH'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['HARVEST'],
            ['WEST'],
            ['SOUTH'],
            ['WEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'],
            ['HARVEST'],
            ['SOUTH'],
            ['HARVEST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['EAST'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['NORTH'],
            ['PASS'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['SOUTH'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['EAST'],
            ['EAST'],
            ['EAST'],
            ['NORTH'],
            ['PASS'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['EAST'],
            ['NORTH'],
            ['EAST'],
            ['NORTH'],
            ['DROP'],
            ['DROP'],
            ['EAST'],
            ['EAST'],
            ['PASS'],
            ['PASS']],
  'market': [['SELL', 'STRAWBERRY', 5]]},
 {'farmer': ['PASS'],
  'hands': [['EAST'],
            ['EAST'],
            ['PASS'],
            ['NORTH'],
            ['PASS'],
            ['PASS'],
            ['EAST'],
            ['EAST'],
            ['PASS'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['EAST'],
            ['EAST'],
            ['PASS'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['DROP'],
            ['DROP'],
            ['PASS'],
            ['PASS']],
  'market': [['SELL', 'STRAWBERRY', 3]]},
 {'farmer': ['PASS'],
  'hands': [['EAST'],
            ['EAST'],
            ['PASS'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['DROP'],
            ['EAST'],
            ['PASS'],
            ['EAST'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS']],
  'market': [['SELL', 'MILK', 3], ['SELL', 'STRAWBERRY', 4]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'],
            ['DROP'],
            ['PASS'],
            ['DROP'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS']],
  'market': [['SELL', 'WOOL', 4], ['SELL', 'MILK', 3], ['SELL', 'STRAWBERRY', 10]]},
 {'farmer': ['PASS'],
  'hands': [['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS']],
  'market': []},
 {'farmer': ['PASS'],
  'hands': [['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS'],
            ['PASS']],
  'market': []}]


SOURCE_MODE = 'sell_guard'
PRODUCTS = ("WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON", "EGG", "MILK", "WOOL", "FERTILIZER")
_METRIC_KEYS = ("actions_changed", "hands_adjusted", "sell_clamped", "sell_dropped", "terminal_rewritten")
_RESCUE_METRICS = {key: 0 for key in _METRIC_KEYS}


def rescue_metrics():
    return dict(_RESCUE_METRICS)


def reset_rescue_metrics():
    for key in _METRIC_KEYS:
        _RESCUE_METRICS[key] = 0
    return rescue_metrics()


def _nonnegative_int(value, default=0):
    try:
        return max(0, int(value))
    except (TypeError, ValueError):
        return default


def _private(obs):
    value = obs.get("private", {}) if isinstance(obs, dict) else {}
    return value if isinstance(value, dict) else {}


def _expected_hands(obs):
    farms = obs.get("farms", []) if isinstance(obs, dict) else []
    player = _nonnegative_int(obs.get("player", 0)) if isinstance(obs, dict) else 0
    if not isinstance(farms, list) or player >= len(farms) or not isinstance(farms[player], dict):
        return 0
    hands = farms[player].get("hands", [])
    return len(hands) if isinstance(hands, list) else 0


def _current_farm(obs):
    farms = obs.get("farms", []) if isinstance(obs, dict) else []
    player = _nonnegative_int(obs.get("player", 0)) if isinstance(obs, dict) else 0
    if not isinstance(farms, list) or player >= len(farms) or not isinstance(farms[player], dict):
        return {}
    return farms[player]


def _unit_is_on_shed(farm, index):
    tiles = farm.get("tiles", [])
    if not isinstance(tiles, list) or not tiles:
        return False
    positions = [farm.get("farmer"), *(farm.get("hands", []) or [])]
    if index >= len(positions):
        return False
    position = positions[index]
    if not isinstance(position, (list, tuple)) or len(position) < 2:
        return False
    x, y = position[0], position[1]
    if not isinstance(x, int) or not isinstance(y, int):
        return False
    board_size = len(tiles)
    half = board_size // 2
    shed_tiles = ((half - 1, half - 1), (half, half - 1), (half - 1, half), (half, half))
    if (x, y) not in shed_tiles or not (0 <= y < len(tiles)):
        return False
    row = tiles[y]
    return isinstance(row, list) and 0 <= x < len(row) and row[x] != "LOCKED"


def _normalize_hands(action, obs):
    expected = _expected_hands(obs)
    original = action.get("hands")
    hands = deepcopy(original) if isinstance(original, list) else []
    hands = hands[:expected]
    hands.extend([["PASS"] for _ in range(expected - len(hands))])
    if hands == original:
        return False
    action["hands"] = hands
    _RESCUE_METRICS["hands_adjusted"] += 1
    return True


def _predicted_shed(action, obs, config=None):
    private = _private(obs)
    raw_shed = private.get("shed", {})
    shed = {
        str(item): _nonnegative_int(quantity)
        for item, quantity in (raw_shed.items() if isinstance(raw_shed, dict) else ())
    }
    inventories = private.get("inventories", [])
    if not isinstance(inventories, list):
        inventories = []
    try:
        shed_capacity = max(0, int((config or {}).get("shedCapacity", 100)))
    except (AttributeError, TypeError, ValueError):
        shed_capacity = 100
    farm = _current_farm(obs)
    units = [action.get("farmer", ["PASS"]), *action.get("hands", [])]
    for index, unit_action in enumerate(units):
        inventory = inventories[index] if index < len(inventories) else {}
        if not isinstance(inventory, dict) or not isinstance(unit_action, list) or not unit_action:
            continue
        if not _unit_is_on_shed(farm, index):
            continue
        if unit_action[0] == "DROP":
            for item, quantity in inventory.items():
                amount = _nonnegative_int(quantity)
                room = max(0, shed_capacity - sum(shed.values()))
                deposited = min(amount, room)
                if deposited:
                    shed[str(item)] = shed.get(str(item), 0) + deposited
        elif unit_action[0] == "PICKUP" and len(unit_action) >= 2:
            item = str(unit_action[1])
            quantity = _nonnegative_int(unit_action[2], 1) if len(unit_action) >= 3 else 1
            shed[item] = max(0, shed.get(item, 0) - quantity)
    return shed


def _guard_sells(market, shed):
    guarded = []
    for original in market if isinstance(market, list) else []:
        if not isinstance(original, list) or not original or original[0] != "SELL":
            guarded.append(deepcopy(original))
            continue
        if len(original) < 2:
            _RESCUE_METRICS["sell_dropped"] += 1
            continue
        item = str(original[1])
        requested = _nonnegative_int(original[2], 1) if len(original) >= 3 else 1
        available = shed.get(item, 0)
        quantity = min(requested, available)
        if quantity <= 0:
            _RESCUE_METRICS["sell_dropped"] += 1
            continue
        order = deepcopy(original)
        if quantity < requested:
            if len(order) >= 3:
                order[2] = quantity
            else:
                order.append(quantity)
            _RESCUE_METRICS["sell_clamped"] += 1
        guarded.append(order)
        shed[item] = available - quantity
    return guarded


def _terminal_market(market, shed, obs):
    preserved = [
        deepcopy(order)
        for order in (market if isinstance(market, list) else [])
        if not (isinstance(order, list) and order and order[0] == "SELL")
    ][:10]
    prices = (obs.get("market", {}) or {}).get("prices", {}) if isinstance(obs, dict) else {}
    prices = prices if isinstance(prices, dict) else {}
    products = [item for item in PRODUCTS if shed.get(item, 0) > 0]
    products.sort(key=lambda item: (-shed[item] * _nonnegative_int(prices.get(item, 0)), item))
    for item in products[: 10 - len(preserved)]:
        preserved.append(["SELL", item, shed[item]])
    return preserved


def agent(obs, config=None):
    try:
        step = int(obs.get("step", 0) or 0)
    except (AttributeError, TypeError, ValueError):
        step = 0
    index = min(max(step, 0), RECORDED_ACTIONS - 1)
    original = deepcopy(SCHEDULE[index])
    action = deepcopy(original)
    _normalize_hands(action, obs)
    shed = _predicted_shed(action, obs, config)
    market = action.get("market", [])
    if SOURCE_MODE == "terminal_guard" and _nonnegative_int(obs.get("day", 0)) >= 29:
        rewritten = _terminal_market(market, shed, obs)
        if rewritten != market:
            _RESCUE_METRICS["terminal_rewritten"] += 1
        action["market"] = rewritten
    else:
        action["market"] = _guard_sells(market, shed)
    if action != original:
        _RESCUE_METRICS["actions_changed"] += 1
    return action


In [ ]:
# Engine-free identity, compile, schedule, and sell-guard checks.
import hashlib
import runpy
from pathlib import Path

payload = Path("main.py").read_bytes()
assert hashlib.sha256(payload).hexdigest().upper() == "81159A18DF3CD1A5CAABE5EDBBA0A597561775F77A49465FA39DF6802A706922"
compile(payload, "main.py", "exec")

policy = runpy.run_path("main.py")
assert len(policy["SCHEDULE"]) == 719
assert policy["SOURCE_MODE"] == "sell_guard"
assert policy["_guard_sells"](
    [["SELL", "WHEAT", 3], ["SELL", "WHEAT", 2]],
    {"WHEAT": 4},
) == [["SELL", "WHEAT", 3], ["SELL", "WHEAT", 1]]
print("source identity, compile, schedule, and sell-guard probe: PASS")


## 6. A benchmark that survives scrutiny

For any change, run a paired matrix rather than one convenient game:

- use the same seeds for baseline and challenger;
- run both physical seats for every seed/opponent pair;
- complete the full 720-turn horizon;
- separate development, confirmation, and holdout seeds;
- record `DONE/DONE`, exceptions, hidden abnormal statuses, and invalid actions;
- report match score, money margin, lower tail, runtime percentiles, and worst case;
- hash source, resolved configuration, cell registry, and raw results before interpretation.

The local separation check for this release used eight consumed seeds, both seats, and exact
source hashes. Each of the two stronger private-source references went 16-0 against this public
source. Across a wider five-public-agent screen, the public source went 0-80. Those results make
the intended strength gap concrete, but the consumed seeds and selected opponents still prevent
them from becoming a general rating forecast.


## 7. Experiments worth trying

Keep the baseline fixed and change only one mechanism at a time:

1. **Guard ablation:** remove `_guard_sells`; count invalid or wasteful sales.
2. **State-triggered fallback:** detect schedule drift and switch to a tiny reactive policy.
3. **End-game liquidation:** replace only the final-day market orders.
4. **Price-aware sizing:** preserve the route but cap sales using observed price and remaining days.
5. **Engine robustness:** repeat identical cells under two pinned versions and explain differences.

If you prefer a fully reactive, shorter starting point, compare this trace baseline with the
[existing public expansion baseline](https://www.kaggle.com/code/yw8837/kaggriculture-strong-expansion-baseline). The useful question is not which
single score is larger; it is which assumptions survive paired both-seat tests.


In [ ]:
import gzip
import hashlib
import io
import json
import tarfile
from pathlib import Path

SOURCE_PATH = Path("main.py")
OUTPUT_NAME = "kaggriculture_fixed_trace_safety_baseline.tar.gz"
EXPECTED_SOURCE_SHA256 = "81159A18DF3CD1A5CAABE5EDBBA0A597561775F77A49465FA39DF6802A706922"
EXPECTED_LICENSE_SHA256 = "C71D239DF91726FC519C6EB72D318EC65820627232B2F796219E87DCF35D0AB4"
EXPECTED_NOTICE_SHA256 = "7C0E05B779269FA8DBEC2AFBAEF928D20223A2E16E3B1DB90DA279922E5471EF"
EXPECTED_ARCHIVE_SHA256 = "5DBD64463878A52ED7C47C8D60BCA3AE9FAC7E29B775E7FADB08F0CC7670A8B6"

source_payload = SOURCE_PATH.read_bytes()
license_payload = b'                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n'
notice_payload = b'# Attribution notice\n\nThis public release contains a policy independently regenerated from a public Kaggriculture replay\nand modified with observation-based hand normalization and impossible-SELL drift guards.\n\n- Upstream work: `Kaggriculture Frontier Lab | High-Score + Visuals`\n- Upstream author: PRVSIYAN\n- Upstream URL: <https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-lab-high-score-visuals>\n- License displayed by upstream: Apache License 2.0\n- Public replay episode: `89256171`\n- Team/player: `radiant-allomancer`, player `0`\n- Replay SHA-256: `ADADDE60EEFB93B56DBE9922B51D5D1970803F89DDF0EEEE67B9890024090405`\n- Released `main.py` SHA-256: `81159A18DF3CD1A5CAABE5EDBBA0A597561775F77A49465FA39DF6802A706922`\n\nThe upstream trajectory is not presented as independently authored strategy. The release source is\nthe exact audited G1 file; it does not incorporate the separately evaluated Kaito, Ray, or Tetsu\nagent bytes.\n\nSee `LICENSE` for the Apache License, Version 2.0.\n\n'
Path("LICENSE").write_bytes(license_payload)
Path("NOTICE.md").write_bytes(notice_payload)
members = {
    "main.py": source_payload,
    "LICENSE": license_payload,
    "NOTICE.md": notice_payload,
}

actual_source_sha256 = hashlib.sha256(source_payload).hexdigest().upper()
assert actual_source_sha256 == EXPECTED_SOURCE_SHA256
assert hashlib.sha256(license_payload).hexdigest().upper() == EXPECTED_LICENSE_SHA256
assert hashlib.sha256(notice_payload).hexdigest().upper() == EXPECTED_NOTICE_SHA256

compressed = io.BytesIO()
with (
    gzip.GzipFile(filename="", mode="wb", fileobj=compressed, mtime=0) as gz,
    tarfile.open(fileobj=gz, mode="w") as archive,
):
    for name, payload in members.items():
        info = tarfile.TarInfo(name)
        info.size = len(payload)
        info.mode = 0o644
        info.mtime = 0
        info.uid = info.gid = 0
        info.uname = info.gname = ""
        archive.addfile(info, io.BytesIO(payload))

output_path = Path("/kaggle/working") / OUTPUT_NAME
output_path.write_bytes(compressed.getvalue())
actual_archive_sha256 = hashlib.sha256(output_path.read_bytes()).hexdigest().upper()
assert actual_archive_sha256 == EXPECTED_ARCHIVE_SHA256
with tarfile.open(output_path, "r:gz") as archive:
    assert archive.getnames() == list(members)
    for member in archive.getmembers():
        assert member.mode == 0o644
        assert member.uid == member.gid == member.mtime == 0
        assert member.uname == member.gname == ""
        assert archive.extractfile(member).read() == members[member.name]

print(json.dumps({
    "artifact": OUTPUT_NAME,
    "source_sha256": actual_source_sha256,
    "member_order": list(members),
    "member_sha256": {
        name: hashlib.sha256(payload).hexdigest().upper()
        for name, payload in members.items()
    },
    "archive_sha256": actual_archive_sha256,
    "archive_bytes": output_path.stat().st_size,
    "competition_submission_performed": False,
}, indent=2))


## 8. Claims and limitations

| Statement | Evidence type | Limit |
|---|---|---|
| The displayed source is the audited G1 artifact | exact SHA-256 and byte-for-byte release check | identity, not strength |
| It lost 32/32 to the two private-source references | local 1.32.6 both-seat screen | consumed seeds; not a rating estimate |
| It lost 80/80 in the five-agent public screen | local 1.32.6 both-seat screen | selected opponents; not a population estimate |
| Expected Kaggle rating | **unknown; no claim made** | only a future public run could observe it |

The policy is open-loop, opponent-blind, tied to one public replay, sensitive to state and engine
drift, and intentionally weaker than the unpublished final agents. Packaging creates an archive;
it does **not** upload Code, publish a Discussion, or submit to the competition.


## 9. License and attribution notice

The full Apache License 2.0 and release notice are included here so the public notebook remains
self-contained. Keep both when redistributing the agent.

<details>
<summary>Apache License 2.0 — full text</summary>

```text
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.
```
</details>

### Release notice

# Attribution notice

This public release contains a policy independently regenerated from a public Kaggriculture replay
and modified with observation-based hand normalization and impossible-SELL drift guards.

- Upstream work: `Kaggriculture Frontier Lab | High-Score + Visuals`
- Upstream author: PRVSIYAN
- Upstream URL: <https://www.kaggle.com/code/prvsiyan/kaggriculture-frontier-lab-high-score-visuals>
- License displayed by upstream: Apache License 2.0
- Public replay episode: `89256171`
- Team/player: `radiant-allomancer`, player `0`
- Replay SHA-256: `ADADDE60EEFB93B56DBE9922B51D5D1970803F89DDF0EEEE67B9890024090405`
- Released `main.py` SHA-256: `81159A18DF3CD1A5CAABE5EDBBA0A597561775F77A49465FA39DF6802A706922`

The upstream trajectory is not presented as independently authored strategy. The release source is
the exact audited G1 file; it does not incorporate the separately evaluated Kaito, Ray, or Tetsu
agent bytes.

See `LICENSE` for the Apache License, Version 2.0.
